# Thinking models task
### By: Nhat Truong
---

# Download packages

In [ ]:
# %pip install python-dotenv # Needed only for local machine
%pip install torch
%pip install camel-ai rouge ollama
%pip install unsloth # For Google Colab
# %pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo # For local machine

  Using cached unsloth-2026.3.4-py3-none-any.whl.metadata (70 kB)
  Using cached unsloth_zoo-2026.3.2-py3-none-any.whl.metadata (32 kB)
  Using cached tyro-1.0.8-py3-none-any.whl.metadata (12 kB)
  Using cached xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl.metadata (1.2 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.7 kB)
  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
  Using cached cut_cross_entropy-25.1.1-py3-none-any.whl.metadata (9.3 kB)
  Using cached msgspec-0.20.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
Using cached unsloth-2026.3.4-py3-none-any.whl (447 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.

In [ ]:
# (FOR LOCAL MACHINE) Reinstall those packages so that it is compatible with unsloth 2026.3.3 and unsloth-zoo 2026.3.1
%pip install datasets==4.3
%pip install transformers==4.57.6
%pip install trl==0.24.0

  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.3.3 requires hf_transfer, which is not installed.
unsloth 2026.3.3 requires trl!=0.19.0,<=0.24.0,>=0.18.2, which is not installed.
unsloth 2026.3.3 requires tyro, which is not installed.
unsloth 2026.3.3 requires xforme

# Import packages

In [ ]:
from ollama import Client
import random
import os
import json
from dotenv import load_dotenv
from datetime import date, datetime, timedelta
from camel.datagen import CoTDataGenerator
from camel.models import ModelFactory
from camel.types import ModelPlatformType, ModelType
from camel.configs import GroqConfig
from camel.agents import ChatAgent

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments, TextStreamer

# Generate questions

In [ ]:
# Set seed for random question generation
random_seed_dayofweek = 314159265
random_seed_hexadecimal = 314159265

In [ ]:
# Question generation functions
def question_generation_dayofweek(n_questions = 5, random_seed=None):
    datetime_range = date(9999, 12, 31) - date(1582, 10, 15) # For simplicity, we consider only the Gregorian calendar, which was used firstly in 15 Oct 1582.
    total_days = datetime_range.days

    random.seed(random_seed)

    questions_answers = {}
    for i in range(n_questions):
        # Create a random date and extract the day of week of that date
        random_days = random.randint(0, total_days)
        random_date = date(1582, 10, 15) + timedelta(days=random_days)
        day_of_week = random_date.strftime('%A')

        question = f'The day of week on {random_date.day} {random_date.strftime('%b')} {random_date.year} (Gregorian calendar) is {day_of_week}. What is the day of week on this day next year?'
        answer = date(random_date.year + 1, random_date.month, random_date.day).strftime('%A') # Day of week of the same day next year
        questions_answers[question] = answer
    return questions_answers

def question_generation_hexadecimal(n_questions = 5, random_seed=None):
    random.seed(random_seed)

    hex_digits = list('0123456789ABCDEF')
    questions_answers = {}
    for i in range(n_questions):
        hex_length = random.randint(2, 10)
        hex_number = ''.join(random.choices(hex_digits, k = hex_length))
        decimal_number = int(hex_number, base=16)

        question = f'How is this hexadecimal number "{hex_number}" converted to decimal number?'
        answer = str(decimal_number)
        questions_answers[question] = answer
    return questions_answers


In [ ]:
questions_answers_dayofweek = question_generation_dayofweek(n_questions=10, random_seed=random_seed_dayofweek)
questions_answers_hexadecimal = question_generation_hexadecimal(n_questions=10, random_seed=random_seed_hexadecimal)
questions_answers_data = {**questions_answers_dayofweek, **questions_answers_hexadecimal}
questions_answers_data

{'The day of week on 2 Nov 8488 (Gregorian calendar) is Tuesday. What is the day of week on this day next year?': 'Wednesday',
 'The day of week on 12 Sep 4871 (Gregorian calendar) is Saturday. What is the day of week on this day next year?': 'Monday',
 'The day of week on 13 Mar 7704 (Gregorian calendar) is Thursday. What is the day of week on this day next year?': 'Friday',
 'The day of week on 11 Jun 8501 (Gregorian calendar) is Saturday. What is the day of week on this day next year?': 'Sunday',
 'The day of week on 24 Mar 5582 (Gregorian calendar) is Wednesday. What is the day of week on this day next year?': 'Thursday',
 'The day of week on 26 Jul 8099 (Gregorian calendar) is Sunday. What is the day of week on this day next year?': 'Monday',
 'The day of week on 29 Mar 2216 (Gregorian calendar) is Friday. What is the day of week on this day next year?': 'Saturday',
 'The day of week on 10 Oct 2062 (Gregorian calendar) is Tuesday. What is the day of week on this day next year?': '

# Generate COT training data

In [ ]:
sys_msg = 'You are good at logical reasoning, critical thinking and producing step-by-step, clear and concise explanations and answers.'

## Create validation datagen to compare chat answer with given correct answer

Groq and Llama 3.1 8b are used for the validation of generated answers. The Llama model is small and good enough for correct validation, and the rate limits of this model in Groq's free plan is 30 requests per minute and 14400 requests per day. More information about rate limits can be found here: https://console.groq.com/docs/rate-limits

In [ ]:
# For local machine
# load_dotenv()
# groq_api_key = os.getenv("groq")

# For Google Colab
from google.colab import userdata
groq_api_key = userdata.get("groq")

os.environ["GROQ_API_KEY"] = groq_api_key

In [ ]:
val_model = ModelFactory.create(
    model_platform=ModelPlatformType.GROQ,
    model_type=ModelType.GROQ_LLAMA_3_1_8B,
    model_config_dict=GroqConfig(temperature=0.2).as_dict(),
)
val_agent = ChatAgent(
    system_message=sys_msg,
    model=val_model,
    message_window_size=10,
)

In [ ]:
val_datagen = CoTDataGenerator(val_agent, golden_answers=questions_answers_data)

## Create COT datgen and generate answers

gpt-oss-120b is OpenAI's powerful open-weight reasoning model. During testing different models, I find that this model is the best one because it almost always return correct answers. It can also programmatically accessible via Ollama or some other providers.

### Option 1: gpt-oss-120b & Ollama (recommended)

This option generates the data quite fast (about 5-30 seconds per question). I am not sure about the rate limit for the free plan, but it should be enough to generate a few tens of question-answer pairs without the rate limit issue.

In [ ]:
# For local machine
# ollama_api_key = os.getenv("ollama")

# For Google Colab
from google.colab import userdata
ollama_api_key = userdata.get("ollama")

os.environ["OLLAMA_API_KEY"] = ollama_api_key

In [ ]:
ollama_client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + os.environ.get('OLLAMA_API_KEY')}
)

In [ ]:
cot_datagen_option = 'ollama' # For naming JSON file when saving data
generated_answers = {}
error_skip = 0 # Number of questions skipped due to exception (e.g. rate limit, empty response)

for question in questions_answers_data.keys():
    print(f"Question: {question}")

    try:
        # Get AI's thought process and answer
        answer = ollama_client.generate(
            'gpt-oss:120b-cloud',
            prompt=question,
            system=sys_msg,
            think='high', # For gpt-oss, valid values are 'low', 'medium' and 'high'
            options={
                'temperature': 0.2,
            }
        )
        generated_answers[question] = answer['response']
        print(f"AI's thought process and answer:\n{answer['response']}")

        # Verify the answer
        is_correct = val_datagen.verify_answer(question, answer['response'])
        print(f"Answer verification result: {'Correct' if is_correct else 'Incorrect'}")
        print("-" * 50)
        print()
    except Exception:
        error_skip += 1
        continue

Question: The day of week on 2 Nov 8488 (Gregorian calendar) is Tuesday. What is the day of week on this day next year?


AI's thought process and answer:
**Answer: Wednesday**

**Why?**  

1. **Leap‑year rule (Gregorian calendar)**  
   - A year is a leap year if it is divisible by 4, except centuries not divisible by 400.  
   - 8488 ÷ 4 = 2122 → divisible by 4.  
   - 8488 is not a century year, so it **is** a leap year (it has 29 Feb).

2. **How the day of week changes from one year to the next**  
   - In a normal year the day of week moves forward 1 day (365 ≡ 1 (mod 7)).  
   - In a leap year it moves forward 2 days **only if the extra day (29 Feb) falls between the two dates**.  
   - The interval from **2 Nov 8488** to **2 Nov 8489** does **not** contain 29 Feb 8488 (that day was *before* 2 Nov) and 8489 is not a leap year, so **no extra day lies in the interval**.

3. **Resulting shift**  
   - Because there is no intervening leap day, the shift is just +1 day.  
   - Starting from Tuesday → **Wednesday**.

(You can verify with Zeller’s congruence: 2 Nov 8488 → h = 3 (Tuesday); 2 Nov 8489 → h = 

AI's thought process and answer:
The day of the week moves forward 1 day in a common year and 2 days in a leap year.  

- 4872 is divisible by 4 and not a century year, so it **is a Gregorian leap year**.  
- Therefore the interval from 12 Sep 4871 to 12 Sep 4872 contains the extra Feb 29, giving a shift of 366 days ≡ 2 (mod 7).

Starting from Saturday, two days later is **Monday**.  

**Answer: Monday.**
Answer verification result: Correct
--------------------------------------------------

Question: The day of week on 13 Mar 7704 (Gregorian calendar) is Thursday. What is the day of week on this day next year?


AI's thought process and answer:
**Friday**.  

*Reason:* 7704 is a Gregorian leap year (divisible by 4 but not by 100). Since March 13 falls after the extra day Feb 29, the interval from 13 Mar 7704 to 13 Mar 7705 contains exactly 365 days, so the weekday advances by one day: Thursday → Friday.
Answer verification result: Correct
--------------------------------------------------

Question: The day of week on 11 Jun 8501 (Gregorian calendar) is Saturday. What is the day of week on this day next year?


AI's thought process and answer:
**Answer: Sunday**

**Reasoning**

1. **Leap‑year rule (Gregorian calendar)**  
   - A year is a leap year if it is divisible by 4, **except** if it is divisible by 100, unless it is also divisible by 400.

2. **Check the two years**  
   - 8501 mod 4 = 1 → **not** a leap year (common year).  
   - 8502 mod 4 = 2 → **not** a leap year (common year).

3. **Effect on the day‑of‑week from one year to the next**  
   - For a date **after Feb 28** (June 11 is well after), the shift in weekday from year Y to year Y+1 depends on whether year Y is a leap year.  
   - If year Y is a common year, the weekday advances by **1 day** (365 ≡ 1 (mod 7)).  
   - If year Y were a leap year, it would advance by **2 days** (366 ≡ 2 (mod 7)).

4. **Apply to our case**  
   - Since 8501 is a common year, the weekday on 11 Jun 8502 is one day later than on 11 Jun 8501.

5. **Compute**  
   - 11 Jun 8501 = **Saturday** → add one day → **Sunday**.

Hence, 11 June 8502 falls on 

...

AI's thought process and answer:
**Step‑by‑step conversion of the hex number `8DB728` to decimal**

| Hex digit | Value (decimal) | Power of 16 | Contribution to total |
|-----------|----------------|------------|-----------------------|
| 8 | 8 | \(16^5\) | \(8 \times 1 048 576 = 8 388 608\) |
| D | 13 | \(16^4\) | \(13 \times 65 536 = 851 968\) |
| B | 11 | \(16^3\) | \(11 \times 4 096 = 45 056\) |
| 7 | 7 | \(16^2\) | \(7 \times 256 = 1 792\) |
| 2 | 2 | \(16^1\) | \(2 \times 16 = 32\) |
| 8 | 8 | \(16^0\) | \(8 \times 1 = 8\) |

Now add all the contributions:

\[
\begin{aligned}
8 388 608 \\
+ 851 968 \\
+ 45 056 \\
+ 1 792 \\
+ 32 \\
+ 8 \\
\hline
= 9 287 464
\end{aligned}
\]

---

### Result
\[
\boxed{8\text{DB}728_{16} = 9\,287\,464_{10}}
\]

---

### Quick sanity check (e.g., with a programming language)

```python
>>> int('8DB728', 16)
9287464
```

The same decimal value is obtained, confirming the manual calculation.
Answer verification result: Correct
-----------------------

AI's thought process and answer:
**Hex → Decimal conversion**

The hexadecimal number  

\[
\text{E3D574}_{16}
\]

has six digits. In base‑16 each position represents a power of 16:

| Position (from right) | Power of 16 | Hex digit | Decimal value of digit |
|----------------------|------------|-----------|------------------------|
| 0 (least‑significant) | \(16^0 = 1\)      | 4 | 4 |
| 1                     | \(16^1 = 16\)     | 7 | 7 |
| 2                     | \(16^2 = 256\)    | 5 | 5 |
| 3                     | \(16^3 = 4096\)   | D | 13 |
| 4                     | \(16^4 = 65536\)  | 3 | 3 |
| 5 (most‑significant) | \(16^5 = 1\,048\,576\) | E | 14 |

Now multiply each digit by its weight and add the results:

\[
\begin{aligned}
\text{E3D574}_{16} &= 
14 \times 1\,048\,576 \;+\;
3 \times 65\,536 \;+\;
13 \times 4\,096 \;+\;
5 \times 256 \;+\;
7 \times 16 \;+\;
4 \times 1 \\[4pt]
&= 14\,680\,064 \;+\; 196\,608 \;+\; 53\,248 \;+\; 1\,280 \;+\; 112 \;+\; 4 \\[4pt]
&= 14\,931\,316 .


AI's thought process and answer:
**Hexadecimal → Decimal conversion**

A hexadecimal (base‑16) number is interpreted as a sum of its digits multiplied by powers of 16, starting with \(16^0\) for the right‑most digit.

The hex number **837** has three digits:

| Position (from right) | Digit | Value of 16ⁿ | Contribution |
|-----------------------|-------|--------------|--------------|
| \(n = 0\) (units)    | 7     | \(16^0 = 1\) | \(7 × 1 = 7\) |
| \(n = 1\) (sixteens) | 3     | \(16^1 = 16\) | \(3 × 16 = 48\) |
| \(n = 2\) (256‑s)    | 8     | \(16^2 = 256\) | \(8 × 256 = 2048\) |

Now add the contributions:

\[
2048 + 48 + 7 = 2103
\]

So  

\[
\boxed{837_{\text{hex}} = 2103_{\text{decimal}}}
\]

---

### Quick verification (division method)

1. Divide 2103 by 16 → quotient = 131, remainder = 7 → right‑most hex digit = 7.  
2. Divide 131 by 16 → quotient = 8, remainder = 3 → next hex digit = 3.  
3. Divide 8 by 16 → quotient = 0, remainder = 8 → left‑most hex digit = 8.  

Reading t

AI's thought process and answer:
**Hex 753 → Decimal**

A hexadecimal (base‑16) number is expanded as a sum of each digit multiplied by a power of 16, starting with the right‑most digit (the “units” place).

| Hex digit | Value | Power of 16 | Contribution |
|-----------|-------|-------------|--------------|
| 7         | 7     | 16² = 256   | 7 × 256 = **1792** |
| 5         | 5     | 16¹ = 16    | 5 × 16  = **80**   |
| 3         | 3     | 16⁰ = 1    | 3 × 1   = **3**    |

Now add the contributions:

\[
1792 + 80 + 3 = 1875
\]

So the hexadecimal number **753₁₆** equals **1875₁₀** in decimal.

---

### Quick check (e.g., with a calculator or programming language)

```python
int('753', 16)   # → 1875
```

or on many calculators you can type `0x753` and it will display `1875`.

---

**Result:**  
\[
\boxed{753_{\text{hex}} = 1875_{\text{decimal}}}
\]
Answer verification result: Correct
--------------------------------------------------

Question: How is this hexadecimal number "BF12F4

AI's thought process and answer:
**Hex → Decimal conversion**

The hexadecimal number  

```
BF12F4A76₁₆
```

has 9 digits.  
Write each digit, replace it by its decimal value, and multiply by the appropriate power of 16 (the right‑most digit is 16⁰, the next is 16¹, …).

| Position (from right) | Hex digit | Decimal value | 16ⁿ               | Contribution (value × 16ⁿ) |
|-----------------------|-----------|---------------|-------------------|-----------------------------|
| 0                     | 6         | 6   | 1                | 6 × 1               = **6** |
| 1                     | 7         | 7   | 16               | 7 × 16              = **112** |
| 2                     | A         | 10  | 256              | 10 × 256            = **2 560** |
| 3                     | 4         | 4   | 4 096            | 4 × 4 096           = **16 384** |
| 4                     | F         | 15  | 65 536           | 15 × 65 536         = **983 040** |
| 5                     | 2         | 

### Option 2: gpt-oss-120b, Camel AI & OpenRouter

In this option, the generation usually takes a bit longer time and the rate limit per day for the OpenRouter's free plan is quite low: only 50 requests/25 questions per day (More information: https://openrouter.ai/docs/api/reference/limits). However, with Camel AI, it is easier to switch to many more models and providers if needed. Also, the responses are quite more deterministic.

In [ ]:
# For local machine
# openrouter_api_key = os.getenv("openrouter")

# For Google Colab
from google.colab import userdata
openrouter_api_key = userdata.get("openrouter")

os.environ["OPENROUTER_API_KEY"] = openrouter_api_key

In [ ]:
# Define the chat model and chat agent
chat_model = ModelFactory.create(
    model_platform=ModelPlatformType.OPENAI_COMPATIBLE_MODEL,
    model_type="openai/gpt-oss-120b:free",
    url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model_config_dict={"temperature": 0.2},
)
chat_agent = ChatAgent(
    system_message=sys_msg,
    model=chat_model,
    message_window_size=10,
    token_limit=131072,
)

In [ ]:
openrouter_datagen = CoTDataGenerator(chat_agent, golden_answers=questions_answers_data)

In [ ]:
cot_datagen_option = 'camel-ai' # For naming JSON file when saving data
generated_answers = {}
error_skip = 0 # Number of questions skipped due to exception (e.g. rate limit, empty response)

for question in questions_answers_data.keys():
    print(f"Question: {question}")

    try:
        # Get AI's thought process and answer
        answer = openrouter_datagen.get_answer(question)
        generated_answers[question] = answer
        print(f"AI's thought process and answer:\n{answer}")

        # Verify the answer & add the question-answer pair if the answer is correct
        is_correct = val_datagen.verify_answer(question, answer)
        print(f"Answer verification result: {'Correct' if is_correct else 'Incorrect'}")
        print("-" * 50)
        print()
    except Exception:
        error_skip += 1
        continue

Question: The day of week on 2 Nov 8488 (Gregorian calendar) is Tuesday. What is the day of week on this day next year?


ERROR:camel.models.model_manager:Error processing with model: <camel.models.openai_compatible_model.OpenAICompatibleModel object at 0x7929ce337e90>
ERROR:camel.models.model_manager:Error processing with model: <camel.models.openai_compatible_model.OpenAICompatibleModel object at 0x7929ce337e90>


AI's thought process and answer:
**1. Analyse the problem requirements**

- We are told that **2 Nov 8488 (Gregorian calendar) is a Tuesday**.  
- We have to find the day of the week for **2 Nov 8489**, i.e. the same calendar date one year later.  
- The Gregorian calendar uses the leap‑year rule:  

  * A year divisible by 4 is a leap year **unless** it is a century year (ends with 00) that is **not** divisible by 400.  

- The day‑of‑week for a given date moves forward by  

  * **+1 day** when the intervening year has **365 days** (a common year).  
  * **+2 days** when the intervening year has **366 days** (a leap year) **and the date is after 29 Feb** of that leap year.

So we must determine whether the year **8488** is a leap year and whether 2 Nov lies after 29 Feb (it does). Then we can compute the weekday shift.

---

**2. List the steps to solve the problem**

| Step | Action |
|------|--------|
| 1 | Check if 8488 is a leap year under Gregorian rules. |
| 2 | Since 2 Nov is 

AI's thought process and answer:
**1. Analyse the problem requirements**

- We are told that **12 September 4871 (Gregorian calendar) is a Saturday**.  
- We must find the day of the week for **the same calendar date one year later: 12 September 4872**.  
- To do this we need to know how many days the calendar “jumps” forward when we go from a given date in one year to the same date in the next year.  
- The jump depends on whether the *next* year is a **leap year** (366 days) or a **common year** (365 days) and on whether the date lies **after February 29** (the extra day in a leap year).

**2. List the steps to solve the problem**

| Step | What we do | Why |
|------|------------|-----|
| 2.1 | Determine whether 4872 is a leap year in the Gregorian calendar. | The size of the year (365 vs 366 days) decides the day‑shift. |
| 2.2 | Check if the given date (12 Sep) is after 29 Feb. | If the date is after the extra day, the leap‑year adds **two** days to the weekday shift; otherwise onl

AI's thought process and answer:
**1. Analyse the problem**

- We are told that **13 Mar 7704** (Gregorian calendar) is a **Thursday**.  
- We have to find the day of the week for **13 Mar 7705** (the same calendar date one year later).  

The only thing that can change the weekday when we move forward exactly one year is whether the year we “skip over” contains an **extra day (29 Feb)**.  
In the Gregorian calendar a year is a leap year when  

* it is divisible by 4 **and**  
* it is **not** a century year unless it is divisible by 400.

If the interval from the first date to the same date next year contains a 29 Feb, the weekday moves forward by **366 ≡ 2 (mod 7)** days; otherwise it moves forward by **365 ≡ 1 (mod 7)** days.

---

**2. List the steps to solve**

1. Determine whether **7704** is a leap year.  
2. Locate the 29 Feb of that year relative to 13 Mar 7704.  
3. Decide whether the interval from 13 Mar 7704 to 13 Mar 7705 includes that extra day.  
4. Compute the weekday s

...

AI's thought process and answer:
**1. Analyse the problem requirements**

- We are given a hexadecimal (base‑16) number: **E3D574**.  
- The task is to convert it to its decimal (base‑10) representation.  
- The answer must be presented together with a clear, step‑by‑step reasoning that shows how the conversion is performed.

**2. List the steps needed to solve the problem**

| Step | What we do | Why it is needed |
|------|------------|------------------|
| 2.1 | Write each hex digit and its decimal value. | Hex digits A‑F represent 10‑15; we need numeric values for the arithmetic. |
| 2.2 | Determine the positional weight of each digit (powers of 16). | In any positional system the right‑most digit has weight \(16^0\), the next \(16^1\), …, the left‑most \(16^{n-1}\). |
| 2.3 | Multiply each digit’s decimal value by its weight. | This gives the contribution of each digit to the overall number. |
| 2.4 | Add all the contributions together. | Summation yields the final decimal value. |

ERROR:camel.models.model_manager:Error processing with model: <camel.models.openai_compatible_model.OpenAICompatibleModel object at 0x7929ce337e90>


AI's thought process and answer:
**1. Analyze the problem requirements**

- **What is given?**  
  A number written in hexadecimal (base‑16) notation: **“837”.**  

- **What is asked?**  
  Convert that hexadecimal representation to its equivalent decimal (base‑10) value.

- **What do we need to show?**  
  A clear, step‑by‑step reasoning that follows the four requested sections:  
  1. Analyse the problem  
  2. List the solution steps  
  3. Execute the steps (perform the calculations)  
  4. State the final decimal result  

- **Key concepts**  
  - In base‑16 each digit represents a power of 16.  
  - The right‑most digit is multiplied by \(16^{0}=1\), the next by \(16^{1}=16\), then \(16^{2}=256\), etc.  
  - Hexadecimal digits 0‑9 keep their usual values; A‑F would represent 10‑15 (not needed here).

**2. List the steps to solve the problem**

| Step | Description |
|------|-------------|
| **2.1** | Identify each digit of the hexadecimal number and its position (units, 16‑s, 256

AI's thought process and answer:
**1. Analyze the problem requirements**

- **What is given?**  
  The hexadecimal (base‑16) number **753**.  
  In hex, the digits are 0‑9 and A‑F, where A=10, B=11, C=12, D=13, E=14, F=15.

- **What is asked?**  
  Convert this hex number to its equivalent decimal (base‑10) value.

- **What must the answer contain?**  
  A clear, step‑by‑step reasoning that shows:
  1. How the hex digits are interpreted with their positional weights (powers of 16).  
  2. The arithmetic performed for each digit.  
  3. The final summed result in decimal.

**2. List the steps to solve the problem**

| Step | Description |
|------|-------------|
| **Step 1** | Write the hex number with each digit’s position (units, 16¹, 16², …). |
| **Step 2** | Replace each hex digit by its decimal value (0‑15). |
| **Step 3** | Multiply each digit’s decimal value by the corresponding power of 16 (its positional weight). |
| **Step 4** | Add all the products together to obtain the decim

AI's thought process and answer:
**1. Analyse the problem**

We have a hexadecimal (base‑16) number  

```
BF12F4A76₁₆
```

and we must express it as a decimal (base‑10) integer.  
In base‑16 each digit represents a value from 0‑15 and its weight is a power of 16:

\[
\text{value}= \sum_{i=0}^{n-1} d_i \times 16^{i}
\]

where \(d_i\) is the digit at position \(i\) counted from the **right‑most** (least‑significant) digit (position 0).

---

**2. List the steps to solve the problem**

| Step | What we do | Why |
|------|------------|-----|
| 2.1 | Write down each hex digit and its decimal equivalent. | To be able to multiply by powers of 16. |
| 2.2 | Determine the power of 16 that belongs to each digit (starting with \(16^0\) on the right). | Gives the weight of each digit. |
| 2.3 | Multiply each digit value by its corresponding power of 16. | Produces the contribution of that digit to the total. |
| 2.4 | Add all the contributions together. | The sum is the decimal representation. |


## Transform and save generated question-answer pairs

In [ ]:
# This function transforms the Q&A data into the Alpaca training data format, which is suitable for supervised fine-tuning (SFT).
# The transformed data is saved to a new JSON file.
def transform_qa_format(generated_qa_pairs):
    # Transform the data
    transformed_data = []
    for question, answer in generated_qa_pairs.items():
        transformed_pair = {
            "instruction": question,
            "input": "",
            "output": answer
        }
        transformed_data.append(transformed_pair)

    # Generate output filename with timestamp
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    output_file = f'transformed-qa_{cot_datagen_option}_{timestamp}.json'

    # Write the transformed data
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(transformed_data, f, ensure_ascii=False, indent=4)

    return output_file, transformed_data

In [ ]:
output_file, transformed_data = transform_qa_format(generated_answers)
print(f"Transformation complete. Output saved to: {output_file}")

Transformation complete. Output saved to: transformed-qa_camel-ai_20260310-155444.json


# Configure Unsloth

In [ ]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    # And also all Instruct versions and Math. Coding verisons!
    # It might take several minutes to download this model
    model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: Will load unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit as a legacy tokenizer.


unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 2026,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
# Presaved dataset file name
# No need to run this if the data transfromation above is done
output_file = "transformed-qa_ollama_20260310-141748.json"

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }
pass

dataset = Dataset.from_json(output_file, split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True)

# Configure and train the model

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 2026,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20 | Num Epochs = 10 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,795,552,768 (1.03% trained)


Step,Training Loss
1,1.685420
2,1.288762
3,1.292713
4,1.336196
5,1.421422
6,1.130023
7,1.057128
8,1.255925
9,1.024567
10,0.905710


# Test the fine-tuned model

Example questions:
- The day of week on 31 Dec 2025 is Wednesday. What is the day of week on this day next year? (Thursday)
- How is this hexadecimal number "1A3F" converted to decimal number? (6719)

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [ ]:
# alpaca_prompt is copied from above
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

# Prepare the input for inference
inputs = tokenizer(
    [
        alpaca_prompt.format(
            "The day of week on 31 Dec 2025 is Wednesday. What is the day of week on this day next year?",  # Instruction
            "",  # Input (empty for this example)
            "",  # Output (leave blank for generation)
        )
    ],
    return_tensors="pt"
).to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 4098)

<｜begin▁of▁sentence｜>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
The day of week on 31 Dec 2025 is Wednesday. What is the day of week on this day next year?

### Input:


### Response:
### **Solution**
The day of week advances by one day each year when the year is not a leap year. Since 2026 is not a leap year, the day of week will advance by one day from Wednesday to Thursday. Therefore, **Thursday** is the day of week on 31 December 2026.

### **Verification**
Alternative approach:
- The date in question is 31 Dec 2025 (Wednesday).
- The year 2025 is not a leap year, so the day of week advances by one day each year.
- Therefore, 31 Dec 2026 is Thursday.

### **Conclusion**
\boxed{\text{Thursday}}<｜end▁of▁sentence｜>


In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

('lora_model/tokenizer_config.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.json')

## Load and use the saved model

In [ ]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

if True:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: Will load unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit as a legacy tokenizer.


unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [ ]:
#  Run the cell containing alpaca_prompt variable above first

inputs = tokenizer(
[
    alpaca_prompt.format(
        'How is this hexadecimal number "1A3F" converted to decimal number?', # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 4098)

<｜begin▁of▁sentence｜>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
How is this hexadecimal number "1A3F" converted to decimal number?

### Input:


### Response:
The hexadecimal number \( 1A3F \) can be converted to decimal by expanding each digit according to its position:

\[
\begin{aligned}
1A3F_{16} &= 1 \times 16^3 + A \times 16^2 + 3 \times 16^1 + F \times 16^0 \\
&= 1 \times 4096 + 10 \times 256 + 3 \times 16 + 15 \times 1 \\
&= 4096 + 2560 + 48 + 15 \\
&= 6719_{10}.
\end{aligned}
\]

### Answer:
\( \boxed{6719} \)<｜end▁of▁sentence｜>


# Personal comments about the testing results
After testing the Deepseek model using around a few similar but new questions and run each question several times, I notice that:
- Before training the model, the model has a high chance to generate wrong answers. Also, the responses are usually long and unnecessary explanation. The model usually cannot generate EOS token and runs infinitively.
- After training the model, it may still generates incorrect results, but the chance to get good results is increase. Also, the responses are concise and there is less chance that the responses are generated infinitively.

The responses should be even better if I can use a larger model for fine-tuning. However, I cannot do it due to limited computational resources.